# Lesson 15A: Hierarchical Reinforcement Learning - Theory

## Introduction

Every algorithm so far picked one primitive action per timestep. Hierarchical RL (HRL) adds **temporal abstraction**: reason in terms of extended, multi-step behaviors ("go to the doorway", "pick up the object") rather than individual primitive actions.

- **Options framework**: the formal machinery for temporally-extended actions
- **Semi-MDPs**: the Bellman equation generalized to actions of variable duration
- **Skill discovery and feudal RL**: how the sub-behaviors get discovered, and how a manager/worker hierarchy divides labor
- **Goal-conditioned RL and HER**: a different axis of abstraction — policies conditioned on *which goal*, and a trick that turns failure into a usable learning signal

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import defaultdict, deque

np.random.seed(42)
torch.manual_seed(42)

## Part 1: Temporal Abstraction and the Options Framework

An **option** (Sutton, Precup & Singh, 1999) is a temporally-extended action: $\langle \mathcal{I}, \pi, \beta \rangle$

- $\mathcal{I}$: the **initiation set** — states where the option can be started
- $\pi$: the **intra-option policy** — what primitive actions to take while the option runs
- $\beta(s)$: the **termination condition** — probability the option ends at state $s$

A high-level policy selects among options the same way a flat policy selects among primitive actions — except each choice now commits to a whole extended sub-behavior. "Navigate to the doorway" is an option: initiation set = anywhere in room 1, intra-option policy = walk toward the doorway, termination = reaching it. This is exactly what makes temporal abstraction valuable: the **high-level decision problem** ("which option next?") can have a far smaller effective search space than the full primitive-action problem, because good options have already absorbed the low-level navigation.

## Part 2: Semi-MDPs, Skill Discovery, and Feudal RL

### Semi-MDPs (SMDPs)

Options turn an MDP into a **semi-Markov decision process**: transitions take a variable number of primitive timesteps $k$, not always 1. The Bellman backup generalizes accordingly — discount by the option's actual duration, not a fixed one step:

$$Q(s, o) \leftarrow Q(s, o) + \alpha \left[ R_{k} + \gamma^k \max_{o'} Q(s', o') - Q(s, o) \right]$$

where $R_k = \sum_{t=0}^{k-1} \gamma^t r_{t+1}$ is the discounted return accumulated *during* the option, and $\gamma^k$ (not $\gamma$) discounts the value of what comes after — because $k$ primitive steps actually elapsed.

### Skill Discovery

Where do options come from? Hand-designing them (as in Part 3 below) doesn't scale. **Skill discovery** methods find useful options automatically — from bottleneck states that many optimal trajectories pass through (doorways are a canonical example), from diversity-maximizing objectives that push a set of learned skills to be mutually distinguishable, or from the eigenvectors of the environment's transition graph (Machado et al.'s "eigenoptions").

### Feudal RL

Feudal RL (Dayan & Hinton, 1993; FeUdal Networks, Vezhnevets et al., 2017) imposes an explicit **manager/worker hierarchy**: a manager sets abstract goals in a learned latent space at a slow timescale, and a worker is rewarded for making progress toward whatever goal the manager currently sets, operating at the fast, primitive-action timescale. Crucially the manager never has to specify *how* — only *what* — decoupling long-horizon credit assignment (the manager's problem) from low-level control (the worker's problem).

In [ ]:
SIZE = 10
DOORWAY = (4, 5)   # the one gap in the wall between room 1 (cols 0-4) and room 2 (cols 5-9)
GOAL = (9, 9)
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def in_bounds(pos):
    return 0 <= pos[0] < SIZE and 0 <= pos[1] < SIZE


def is_wall(pos):
    return pos[1] == 5 and pos != DOORWAY


def flat_step(pos, action):
    di, dj = DELTAS[action]
    next_pos = (pos[0] + di, pos[1] + dj)
    if not in_bounds(next_pos) or is_wall(next_pos):
        next_pos = pos
    done = next_pos == GOAL
    reward = 10.0 if done else -1.0
    return next_pos, reward, done


def run_flat_qlearning(n_episodes=800, max_steps=200, alpha=0.3, gamma=0.95, epsilon=0.15):
    """Ordinary Q-learning over the 4 primitive actions -- has to discover the doorway itself."""
    Q = defaultdict(lambda: np.zeros(4))
    steps_history = []
    for _ in range(n_episodes):
        pos = (0, 0)
        for t in range(max_steps):
            action = np.random.randint(4) if np.random.rand() < epsilon else int(np.argmax(Q[pos]))
            next_pos, reward, done = flat_step(pos, action)
            Q[pos][action] += alpha * (reward + gamma * np.max(Q[next_pos]) - Q[pos][action])
            pos = next_pos
            if done:
                break
        steps_history.append(t + 1)
    return steps_history


# Two hand-designed options: "go to the doorway", "go to the goal". Each one's intra-option
# policy is a simple greedy walk toward its target -- all the low-level navigation is baked in.
OPTIONS = {'goto_doorway': DOORWAY, 'goto_goal': GOAL}
OPTION_NAMES = list(OPTIONS.keys())


def greedy_action_toward(pos, target):
    best_action, best_distance = None, np.inf
    for action in range(4):
        next_pos, _, _ = flat_step(pos, action)
        distance = abs(next_pos[0] - target[0]) + abs(next_pos[1] - target[1])
        if distance < best_distance:
            best_distance, best_action = distance, action
    return best_action


def run_option(pos, option_name, max_option_steps=30, gamma=0.95):
    target = OPTIONS[option_name]
    total_reward, discount, steps, done = 0.0, 1.0, 0, False
    while pos != target and steps < max_option_steps:
        action = greedy_action_toward(pos, target)
        pos, reward, done = flat_step(pos, action)
        total_reward += discount * reward
        discount *= gamma
        steps += 1
        if done:
            break
    return pos, total_reward, done, steps


def run_smdp_qlearning(n_episodes=50, max_options=10, alpha=0.3, gamma=0.95, epsilon=0.15):
    """SMDP Q-learning over the 2 OPTIONS, not the 4 primitive actions."""
    Q = defaultdict(lambda: np.zeros(len(OPTION_NAMES)))
    primitive_steps_history = []
    for _ in range(n_episodes):
        pos = (0, 0)
        total_primitive_steps = 0
        for _ in range(max_options):
            option_idx = np.random.randint(len(OPTION_NAMES)) if np.random.rand() < epsilon else int(np.argmax(Q[pos]))
            option_name = OPTION_NAMES[option_idx]
            next_pos, option_reward, done, option_duration = run_option(pos, option_name)
            total_primitive_steps += option_duration
            # SMDP Bellman backup: discount the bootstrap by gamma^k, k = option duration, not 1
            Q[pos][option_idx] += alpha * (
                option_reward + (gamma ** option_duration) * np.max(Q[next_pos]) - Q[pos][option_idx]
            )
            pos = next_pos
            if done:
                break
        primitive_steps_history.append(total_primitive_steps)
    return primitive_steps_history


flat_steps = run_flat_qlearning()
smdp_steps = run_smdp_qlearning()

print(f"Flat Q-learning (4 primitive actions):  first 20 eps mean {np.mean(flat_steps[:20]):.0f} primitive steps, "
      f"last 20 eps mean {np.mean(flat_steps[-20:]):.0f}")
print(f"SMDP Q-learning (2 hand-designed options): first 5 eps mean {np.mean(smdp_steps[:5]):.0f} primitive steps, "
      f"last 5 eps mean {np.mean(smdp_steps[-5:]):.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(flat_steps)
axes[0].set_title(f'Flat Q-learning: {len(flat_steps)} episodes to converge')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Primitive steps to goal')

axes[1].plot(smdp_steps)
axes[1].set_title(f'SMDP options Q-learning: {len(smdp_steps)} episodes to converge')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Primitive steps to goal')
plt.tight_layout()
plt.show()

print("With good options already absorbing the low-level navigation, the HIGH-LEVEL decision")
print("problem has just 2 choices per state instead of 4 -- and one of the two is almost always")
print("obviously wrong, so the options agent converges in a handful of episodes rather than hundreds.")

## Part 3: Goal-Conditioned RL and Hindsight Experience Replay

**Goal-conditioned RL** trains a single policy $\pi(a \mid s, g)$ conditioned on a *goal* $g$, rather than one policy per fixed task — one network that generalizes across "go to the doorway", "go to the goal", or any other reachable target. This unifies the options above into a single learned policy rather than a table of hand-designed sub-policies.

**Hindsight Experience Replay** (Andrychowicz et al., 2017) solves goal-conditioned RL's worst failure mode: with a sparse "did I reach exactly this goal?" reward, a randomly-initialized policy almost never succeeds, so almost every episode teaches it nothing at all. HER's fix is almost embarrassingly simple: **after** a failed episode, relabel it — pretend the goal was whatever state the agent *actually* ended up in, not the one it was commanded to reach. That failed trajectory becomes a *successful* one for the achieved goal, giving the replay buffer a nonzero-reward learning signal on every single episode, regardless of what was originally commanded.

In [ ]:
N_BITS = 10


class BitFlipEnv:
    """The canonical HER toy problem (Andrychowicz et al., 2017): flip one bit per step,
    try to match a target n-bit vector. Reward is 0 only at the exact goal, -1 otherwise --
    a textbook sparse-reward, goal-conditioned task."""

    def __init__(self, n_bits=N_BITS):
        self.n_bits = n_bits

    def reset(self):
        self.state = np.random.randint(0, 2, self.n_bits).astype(np.float32)
        self.goal = np.random.randint(0, 2, self.n_bits).astype(np.float32)
        return self.state.copy(), self.goal.copy()

    def step(self, action):
        self.state[action] = 1 - self.state[action]
        done = np.array_equal(self.state, self.goal)
        reward = 0.0 if done else -1.0
        return self.state.copy(), reward, done


def encode(state, goal):
    return np.concatenate([state, goal]).astype(np.float32)


class GoalConditionedQNet(nn.Module):
    def __init__(self, n_bits, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 * n_bits, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_bits))

    def forward(self, x):
        return self.net(x)


def train(use_her, n_episodes=400, gamma=0.98, batch_size=128, lr=1e-3, epsilon_start=0.3):
    env = BitFlipEnv()
    net = GoalConditionedQNet(N_BITS)
    target_net = GoalConditionedQNet(N_BITS)
    target_net.load_state_dict(net.state_dict())
    optimizer = optim.Adam(net.parameters(), lr=lr)
    buffer = deque(maxlen=50000)
    success_history = []

    for ep in range(n_episodes):
        epsilon = max(0.05, epsilon_start * (1 - ep / n_episodes))
        state, goal = env.reset()
        trajectory = []
        success = False
        for _ in range(N_BITS):
            enc = encode(state, goal)
            if np.random.rand() < epsilon:
                action = np.random.randint(N_BITS)
            else:
                with torch.no_grad():
                    action = int(torch.argmax(net(torch.as_tensor(enc))).item())
            next_state, reward, done = env.step(action)
            trajectory.append((state.copy(), action, reward, next_state.copy(), goal.copy(), done))
            state = next_state
            if done:
                success = True
                break
        success_history.append(success)

        # Standard replay: every transition with its ORIGINAL commanded goal.
        for s, a, r, ns, g, d in trajectory:
            buffer.append((encode(s, g), a, r, encode(ns, g), d))

        if use_her:
            # HER "final" strategy: relabel the whole trajectory with the state it ACTUALLY
            # ended in as the goal -- turns a failed episode into a successful one.
            achieved_goal = trajectory[-1][3]
            for s, a, r, ns, g, d in trajectory:
                relabel_done = np.array_equal(ns, achieved_goal)
                relabel_reward = 0.0 if relabel_done else -1.0
                buffer.append((encode(s, achieved_goal), a, relabel_reward, encode(ns, achieved_goal), relabel_done))

        if len(buffer) >= batch_size:
            for _ in range(4):
                batch = [buffer[i] for i in np.random.randint(0, len(buffer), batch_size)]
                S = torch.as_tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
                A = torch.as_tensor([b[1] for b in batch], dtype=torch.long)
                R = torch.as_tensor([b[2] for b in batch], dtype=torch.float32)
                NS = torch.as_tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
                D = torch.as_tensor([float(b[4]) for b in batch], dtype=torch.float32)
                q = net(S).gather(1, A.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    target = R + gamma * (1 - D) * torch.max(target_net(NS), dim=1).values
                loss = nn.functional.mse_loss(q, target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        if ep % 20 == 0:
            target_net.load_state_dict(net.state_dict())
    return success_history


no_her_successes = train(use_her=False)
with_her_successes = train(use_her=True)

print(f"Without HER: {sum(no_her_successes)} / {len(no_her_successes)} episodes reached the exact goal")
print(f"With HER:    {sum(with_her_successes)} / {len(with_her_successes)} episodes reached the exact goal")

In [ ]:
window = 50
fig, ax = plt.subplots(figsize=(8, 4))
for successes, label in [(no_her_successes, 'without HER'), (with_her_successes, 'with HER')]:
    rate = np.convolve(successes, np.ones(window) / window, mode='valid')
    ax.plot(rate, label=label)
ax.set_xlabel('Episode')
ax.set_ylabel(f'Success rate (window={window})')
ax.set_title(f'{N_BITS}-bit flipping: HER turns failed episodes into a usable learning signal')
ax.legend()
plt.tight_layout()
plt.show()

## Key Concepts

1. **Options**: temporally-extended actions — $\langle \mathcal{I}, \pi, \beta \rangle$ — turning an MDP into a semi-MDP where transitions take a variable number of steps
2. **SMDP Bellman backup**: discount the bootstrap by $\gamma^k$ (the option's actual duration $k$), not a fixed single step
3. **Skill discovery**: automatically finding useful options (bottleneck states, diversity objectives, eigenoptions) instead of hand-designing them
4. **Feudal RL**: manager sets abstract goals at a slow timescale, worker pursues them at the fast primitive-action timescale — decouples long-horizon credit assignment from low-level control
5. **Goal-conditioned RL**: one policy $\pi(a \mid s, g)$ generalizing across goals, rather than one policy per task
6. **HER**: relabel failed trajectories with their actually-achieved outcome as the goal — turns every episode into a learning signal, which this notebook's own bit-flipping experiment showed going from a 400-episode-long standstill to full convergence